# Huấn luyện YOLO26n (Construction-PPE + BAC_HIEN) trên Google Colab

**Trước khi chạy**
1. Tải `ppe-train-combined.zip` (417 MB) lên Google Drive, đặt tại `MyDrive/helmet/ppe-train-combined.zip`.
2. Chọn **Runtime → Change runtime type → T4 GPU**.
3. Bấm **Runtime → Run all**, hoặc chạy lần lượt từng ô. Lần đầu Colab sẽ hỏi quyền truy cập Drive: chọn Cho phép.

Kết quả (trọng số, `results.csv`, `results.png`) lưu thẳng vào `MyDrive/helmet/runs/helmet_yolo26n_v3/`.
Nếu Colab bị ngắt giữa chừng, **mở lại và chạy lại toàn bộ**: ô huấn luyện tự nối tiếp từ `last.pt`.

Thời gian ước tính: khoảng 1,5–3 giờ (Colab chỉ có 2 CPU nên nạp ảnh là điểm chậm).

In [ ]:
# 1. Kiểm tra GPU
import torch
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Cài Ultralytics (YOLO26 cần bản >= 8.4)
!pip -q install -U ultralytics
import ultralytics; ultralytics.checks()

In [ ]:
# 3. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Giải nén dữ liệu và kiểm tra số lượng
!unzip -q -o /content/drive/MyDrive/helmet/ppe-train-combined.zip -d /content
import os
for s in ('train', 'val'):
    n = len(os.listdir(f'/content/ppe-train-combined/images/{s}'))
    print(s, n, 'ảnh')
print(open('/content/ppe-train-combined/data.yaml').read())
# Kỳ vọng: train 5132 ảnh, val 503 ảnh, 11 lớp

## 5. Huấn luyện

Cấu hình giống `scripts/train.py` của dự án: AdamW, lr0 = 0,000667, imgsz 640, batch 16, seed 0.
Có thể đổi `EPOCHS` nếu thiếu thời gian (ví dụ 60). Nếu báo hết bộ nhớ GPU, đổi `BATCH = 8`.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

DATA    = '/content/ppe-train-combined/data.yaml'
PROJECT = '/content/drive/MyDrive/helmet/runs'   # lưu thẳng vào Drive để không mất khi Colab ngắt
NAME    = 'helmet_yolo26n_v3'
EPOCHS, PATIENCE, BATCH = 80, 15, 16

last = Path(PROJECT) / NAME / 'weights' / 'last.pt'
if last.exists():
    print('Tìm thấy', last, '-> tiếp tục huấn luyện')
    model = YOLO(str(last))
    model.train(resume=True)
else:
    model = YOLO('yolo26n.pt')   # trọng số tiền huấn luyện, tự tải về
    model.train(
        data=DATA, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, imgsz=640, device=0,
        project=PROJECT, name=NAME, exist_ok=True,
        optimizer='AdamW', lr0=0.000667, lrf=0.01, momentum=0.937, weight_decay=0.0005,
        warmup_epochs=3.0, close_mosaic=10, seed=0, deterministic=True, amp=True,
        workers=2, plots=True,
    )

## 6. Đánh giá nhanh trên tập val (3 lớp hệ thống dùng)

In [ ]:
best = Path(PROJECT) / NAME / 'weights' / 'best.pt'
m = YOLO(str(best)).val(data=DATA, split='val', imgsz=640, device=0, plots=False)
names = {0: 'helmet', 6: 'Person', 7: 'no_helmet'}
idx = {int(c): i for i, c in enumerate(m.box.ap_class_index)}
for cid, n in names.items():
    if cid in idx:
        p, r, ap50, ap = m.box.class_result(idx[cid])
        print(f'{n:10s} P={p:.3f} R={r:.3f} mAP50={ap50:.3f} mAP50-95={ap:.3f}')

In [ ]:
# 7. Biểu đồ quá trình huấn luyện (đưa vào báo cáo)
from IPython.display import Image, display
display(Image(filename=f'{PROJECT}/{NAME}/results.png'))

In [ ]:
# 8. Gom file cần gửi lại vào một thư mục trên Drive
import shutil
out = Path('/content/drive/MyDrive/helmet/yolo26n_result')
out.mkdir(parents=True, exist_ok=True)
shutil.copy(best, out / 'best.pt')
for f in ('results.csv', 'results.png', 'args.yaml'):
    shutil.copy(Path(PROJECT) / NAME / f, out / f)
print(sorted(p.name for p in out.iterdir()), '->', out)
# Tải 4 file này về máy: best.pt, results.csv, results.png, args.yaml